KTO implementation: what actually goes into the loss

Now we turn the theory into the exact pieces we'd calculate.

For each training example we have:

prompt x
response y
label = 👍 or 👎
Step 1 — Get the policy log-probability

Just like DPO, we calculate:

$$ \log \pi_\theta(y|x) $$

Meaning:

How likely is the current model to produce this exact response?

We already know how to calculate this from our DPO implementation: tokenize the conversation → forward pass → log_softmax → gather response-token probabilities → sum them.

Step 2 — Get the reference log-probability

Using the frozen reference model:

$$ \log \pi_{ref}(y|x) $$

Same response, same calculation.

So now:

policy log-prob     = -120
reference log-prob  = -130

The policy likes this response 10 log-prob units more than the reference.

Step 3 — Calculate the relative preference signal

The core comparison is:

$$ \log\frac{\pi_\theta(y|x)} {\pi_{ref}(y|x)} $$

Using the log identity:

$$ \log\frac{\pi_\theta}{\pi_{ref}} = \log\pi_\theta-\log\pi_{ref} $$

So:

-120 - (-130)
= +10

That +10 means:

Current model increased the response's relative probability compared with the reference.

A negative number means it decreased it.

Step 4 — Apply the KL/reference offset

KTO doesn't want the model to blindly maximize that number.

It introduces a reference point, commonly represented as \(z_{ref}\):

$$ r = \beta \left[ (\log\pi_\theta-\log\pi_{ref})-z_{ref} \right] $$

So the model is effectively asking:

“Did I move toward this response enough, relative to the reference behavior?”

Step 5 — Apply the label

Now the feedback determines the direction:

👍  → push r upward
👎  → push r downward

That's the entire conceptual mechanism.

And notice something important:

There is still no reward model.

We directly use:

policy probability
       ↓
reference probability
       ↓
relative signal
       ↓
binary feedback
       ↓
gradient update

That's why KTO fits nicely beside DPO in the preference-learning family.